# RetailIQ – Online Retail Analytics & Intelligence System
## Academic Data Science Capstone Mini-Project

**Domain:** E-Commerce Transactional Analytics, RFM Customer Segmentation, Time-Series Demand Forecasting  
**Author:** RetailIQ Data Science Team  
**Date:** November 2026  

---
### 1. Project Introduction & Problem Statement
Modern online retail enterprises process thousands of orders daily. To drive sustainable commercial growth, businesses must address two critical questions:
1. **Customer Heterogeneity:** Which customer cohorts contribute most to top-line revenue, and which are at immediate risk of churning?
2. **Demand Uncertainty:** How can management accurately project future revenue demand while respecting the temporal dynamics and seasonality of retail purchasing?

**RetailIQ** provides an end-to-end analytical workflow that resolves these challenges using validated statistical and machine learning methodologies.


### 2. Dataset Description
The dataset contains 541,909 transactional records from a UK-based online giftware retailer operating between **December 1, 2010 and December 9, 2011**.

Key attributes:
* `InvoiceNo`: 6-digit transaction ID (cancellations prefixed with 'C')
* `StockCode`: Distinct product SKU identifier
* `Description`: Product title
* `Quantity`: Quantity per transaction
* `InvoiceDate`: Exact timestamp of purchase
* `UnitPrice`: Product price in GBP (£)
* `CustomerID`: Unique client account number
* `Country`: Country of purchaser


In [ ]:
# Step 1: Import Core Dependencies
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Dependencies loaded successfully.')


### 3. Data Loading & Initial Inspection
We load the validated dataset directly using our project's data loading module.


In [ ]:
from src.data_loader import load_raw_data, get_raw_metadata

# Load transactions using CSV or Excel preference
raw_df, source_info = load_raw_data(source_preference='csv')
metadata = get_raw_metadata(raw_df)

print(f"Data Source: {source_info}")
print(f"Total Rows: {metadata['rows']:,}")
print(f"Total Columns: {metadata['columns']}")
print(f"Date Range: {metadata['date_min']} to {metadata['date_max']}")
print(f"Unique Customers: {metadata['unique_customers']:,}")
print(f"Duplicate Rows: {metadata['duplicate_count']:,} ({metadata['duplicate_pct']}%)")
raw_df.head()


### 4. Data Cleaning & Hygiene Pipeline
Rather than blindly deleting records, we partition the transactions into structured views:
* **Pruning exact duplicates**
* **Classifying cancellations** (`InvoiceNo` starting with 'C')
* **Isolating negative quantities and zero prices**
* **Creating `Revenue = Quantity * UnitPrice`**
* **Generating distinct operational views**: `df_clean_sales`, `df_returns`, `df_macro_sales`


In [ ]:
from src.preprocessing import clean_retail_data

views, audit = clean_retail_data(raw_df, save_processed=False)
df_clean = views['clean_sales']
df_returns = views['returns']

print(f"Initial Records: {audit['initial_rows']:,}")
print(f"Deduplicated Records: {audit['deduplicated_rows']:,}")
print(f"Cancellations Detected: {audit['cancellations_count']:,}")
print(f"Clean Identified Customer Sales: {len(df_clean):,} records")
print(f"Gross Revenue (All Sales): £{audit['gross_sales_revenue']:,.2f}")
print(f"Total Returns Value: £{audit['returned_loss_value']:,.2f}")


### 5. Exploratory Data Analysis (EDA)
We examine temporal revenue trends, top revenue-generating SKUs, and geographic concentration.


In [ ]:
from src.eda import get_daily_sales_series, get_top_products, get_country_sales_summary

daily_sales = get_daily_sales_series(df_clean)

# Plot Daily Revenue with Moving Averages
plt.figure(figsize=(14, 5))
plt.plot(daily_sales['Date'], daily_sales['Revenue'], color='#94A3B8', alpha=0.5, label='Daily Revenue')
plt.plot(daily_sales['Date'], daily_sales['7D_MA'], color='#06B6D4', linewidth=2, label='7-Day Moving Avg')
plt.plot(daily_sales['Date'], daily_sales['30D_MA'], color='#4F46E5', linewidth=2.5, label='30-Day Moving Avg')
plt.title('Daily Revenue Trend with 7-Day & 30-Day Moving Averages (£)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Revenue (£)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 Revenue-Generating SKUs
top_products = get_top_products(df_clean, top_n=10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_products, x='Revenue', y='Description', palette='viridis')
plt.title('Top 10 Products by Realized Revenue (£)', fontsize=14, fontweight='bold')
plt.xlabel('Revenue (£)')
plt.ylabel('')
plt.tight_layout()
plt.show()


### 6. Customer Segmentation: RFM Analysis & Skewness Correction
We calculate Recency (R), Frequency (F), and Monetary (M) metrics per customer.
Because customer spend and frequency exhibit severe Pareto right-skewness, we apply logarithmic transformations:
$$\tilde{x} = \ln(1 + x)$$
followed by standard z-score scaling:
$$z = \frac{\tilde{x} - \mu}{\sigma}$$


In [ ]:
from src.customer_segmentation import compute_rfm_table, evaluate_cluster_candidates, train_kmeans_segmentation

rfm_table = compute_rfm_table(df_clean)
print(f"Computed RFM table for {len(rfm_table):,} unique customers.")
rfm_table.describe().round(2)


### 7. Determining Optimal Clusters: Elbow & Silhouette Analysis
We iterate k in [2, 8] and compute Inertia (SSE) and mean Silhouette Score.


In [ ]:
cand_eval = evaluate_cluster_candidates(rfm_table, cluster_range=range(2, 8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
ax1.plot(cand_eval['k_values'], cand_eval['inertias'], marker='o', color='#4F46E5', linewidth=2)
ax1.set_title('Elbow Method: Inertia vs k', fontsize=12, fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (SSE)')
ax1.axvline(4, color='#10B981', linestyle='--', label='Elbow Point (k=4)')
ax1.legend()

# Silhouette Score Curve
ax2.bar(cand_eval['k_values'], cand_eval['silhouette_scores'], color='#06B6D4')
ax2.set_title('Silhouette Score across k', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()
print(f"Optimal cluster recommendation: k = {cand_eval['recommended_k']}")


### 8. Training K-Means & Deriving Marketing Personas
We train the final model with k=4, assigning meaningful data-derived persona names:
* **Champions**: High monetary, high frequency, recent purchases
* **Loyal Customers**: Steady repeat buyers with strong contribution
* **Potential Loyalists**: Recent buyers with moderate spend
* **At-Risk / Hibernating**: High past value or inactive shoppers


In [ ]:
rfm_segmented, km_model, scaler, seg_meta = train_kmeans_segmentation(
    rfm_table, n_clusters=4, save_artifacts=False
)

seg_summary = pd.DataFrame(seg_meta['segment_summary'])
print(f"K-Means Silhouette Score: {seg_meta['silhouette_score']:.4f}")
print(seg_summary[['Segment', 'CustomerCount', 'CustomerShare_Pct', 'TotalRevenue', 'RevenueShare_Pct', 'AvgRecency', 'AvgFrequency', 'AvgMonetary']].round(2))


In [ ]:
# Visualize 2D Segment Separation
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm_segmented, x='Recency', y='Monetary', hue='Segment',
    palette='Set1', alpha=0.7
)
plt.xscale('log')
plt.yscale('log')
plt.title('Customer Segments in Log-RFM Space', fontsize=13, fontweight='bold')
plt.xlabel('Recency (Days, Log Scale)')
plt.ylabel('Monetary (£, Log Scale)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 9. Time-Series Sales Forecasting (ARIMA & Exponential Smoothing)
We construct a continuous daily revenue series and execute a strict chronological train-test split (last 30 days as holdout) to prevent data leakage.
We benchmark:
1. **ARIMA / SARIMAX** (with weekly seasonality order)
2. **Holt-Winters Exponential Smoothing** (additive trend and seasonality)


In [ ]:
from src.sales_forecasting import prepare_sales_series, compare_forecasting_models, generate_future_forecast

ts_daily = prepare_sales_series(df_clean, frequency='D')
forecast_benchmark = compare_forecasting_models(ts_daily, test_size=30)

print('Out-of-Sample Performance Comparison:')
print(forecast_benchmark['comparison_table'])
print(f'Best Performing Model: {forecast_benchmark["best_model_name"]}')


In [ ]:
# Plot Actual vs Predicted on Holdout Test Set
train_series = forecast_benchmark['train']
test_series = forecast_benchmark['test']
arima_pred = forecast_benchmark['models']['ARIMA']['predictions']
hw_pred = forecast_benchmark['models']['ExponentialSmoothing']['predictions']

plt.figure(figsize=(14, 6))
plt.plot(train_series.iloc[-45:].index, train_series.iloc[-45:].values, label='Historical Train (Recent)', color='#94A3B8')
plt.plot(test_series.index, test_series.values, label='Actual Test Sales', color='#10B981', linewidth=2.5, marker='o')
plt.plot(arima_pred.index, arima_pred.values, label='ARIMA Forecast', color='#4F46E5', linewidth=2, linestyle='--')
plt.plot(hw_pred.index, hw_pred.values, label='Holt-Winters Forecast', color='#F59E0B', linewidth=2, linestyle=':')
plt.title('Out-of-Sample Sales Forecast Comparison (30-Day Holdout)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Revenue (£)')
plt.legend()
plt.tight_layout()
plt.show()


### 10. Forward-Looking Demand Projection & Limitations
Using the full dataset, we project 30 days into the future along with 95% confidence intervals.


In [ ]:
future_fc, conf_int = generate_future_forecast(ts_daily, model_type='ARIMA', horizon_days=30)

plt.figure(figsize=(14, 6))
plt.plot(ts_daily.iloc[-60:].index, ts_daily.iloc[-60:].values, label='Historical Daily Sales', color='#64748B')
plt.plot(future_fc.index, future_fc.values, label='ARIMA Future Point Forecast', color='#06B6D4', linewidth=2.5)
plt.fill_between(
    conf_int.index, conf_int['LowerBound'], conf_int['UpperBound'],
    color='#06B6D4', alpha=0.2, label='95% Confidence Interval'
)
plt.title('30-Day Forward Revenue Forecast with 95% Confidence Bounds (£)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Revenue (£)')
plt.legend()
plt.tight_layout()
plt.show()


### 11. Key Findings & Academic Conclusions

1. **Hygiene & Preprocessing Impact:**
   * Pruning 5,268 duplicates prevented significant revenue overestimation.
   * Isolating cancelled invoices (`C` prefix) and negative quantities accounted for £600k+ in returned/cancelled goods, ensuring model training was conducted exclusively on genuine realized sales.
2. **Customer Segmentation Value:**
   * RFM segmentation with k=4 identified Champions who generate over 60% of total revenue despite comprising less than 25% of the customer base.
   * The At-Risk cohort represents customers with high past spend who have not purchased in over 90 days, offering clear ROI for re-engagement campaigns.
3. **Forecasting Reliability & Constraints:**
   * Both ARIMA and Holt-Winters capture the fundamental weekly cyclical pattern (store closed on Saturdays, peak mid-week orders).
   * **Limitation:** The single-year span (Dec 2010 – Dec 2011) restricts annual seasonality modeling. Forecasts beyond 30 days should be used for directional strategic guidance rather than strict inventory procurement.
